In [12]:
import pandas as pd
import matplotlib.pyplot as plt

import os
import pandas as pd

# List all CSV files in the pbp folder
pbp_folder = 'pbp'
csv_files = [f for f in os.listdir(pbp_folder) if f.endswith('.csv')]

# Sort files by year (assuming filename format pbpYYYY.csv)
csv_files.sort()

# Read and concatenate all CSV files
dfs = []
for file in csv_files:
    file_path = os.path.join(pbp_folder, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

# Concatenate all dataframes
pbp_data = pd.concat(dfs, ignore_index=True)

# Display the shape of the combined dataframe
print(f"Combined PBP data shape: {pbp_data.shape}")
pbp_data.head()



Combined PBP data shape: (16215625, 16)


,gameid,period,clock,h_pts,a_pts,team,playerid,player,type,subtype,result,x,y,dist,desc,season
0,29600001,1,PT12M00.00S,0.0,0.0,NaN,0,NaN,period,start,NaN,0,0,0,Start of 1st Period (11:15 PM EST),1997
1,29600001,1,PT12M00.00S,0.0,0.0,BOS,442,P. Ellison,Jump Ball,NaN,NaN,0,0,0,Jump Ball Ellison vs. Longley: Tip to Harper,1997
2,29600001,1,PT11M39.00S,0.0,2.0,CHI,23,D. Rodman,Made Shot,Layup Shot,Made,0,0,0,Rodman Layup (2 PTS) (Longley 1 AST),1997
3,29600001,1,PT11M39.00S,0.0,0.0,BOS,442,P. Ellison,Foul,Shooting,NaN,0,0,0,Ellison S.FOUL (P1.T1),1997
4,29600001,1,PT11M39.00S,0.0,3.0,CHI,23,D. Rodman,Free Throw,Free Throw 1 of 1,NaN,0,0,0,Rodman Free Throw 1 of 1 (3 PTS),1997


In [21]:
# 1. Filter to Curry only, clean outcome
curry_shots = pbp_data[pbp_data["player"] == "S. Curry"].copy()
curry_shots = curry_shots.sort_values(["gameid", "period", "clock"]).reset_index(drop=True)
curry_shots = pbp_data[
    (pbp_data["player"] == "S. Curry") & 
    (pbp_data["type"].isin(["Made Shot", "Missed Shot"]))
].copy()
curry_shots["outcome"] = (curry_shots["type"] == "Made Shot").astype(int)



# 3. Build lags — reset at game boundary via groupby
curry_shots["lag1"] = curry_shots.groupby("gameid")["outcome"].shift(1)
curry_shots["lag2"] = curry_shots.groupby("gameid")["outcome"].shift(2)
curry_shots["lag3"] = curry_shots.groupby("gameid")["outcome"].shift(3)

# 4. Drop rows where any lag is missing (first 3 shots of each game)
model_data = curry_shots.dropna(subset=["lag1", "lag2", "lag3"]).copy()
model_data[["lag1", "lag2", "lag3"]] = model_data[["lag1", "lag2", "lag3"]].astype(int)

print(f"Total shots: {len(curry_shots)}")
print(f"Shots available for modeling: {len(model_data)}")
print(f"Shots dropped (first 3 per game): {len(curry_shots) - len(model_data)}")
model_data.head(10)
curry_shots.to_csv("curry_shots.csv", index=False)
model_data.to_csv("curry_model_data.csv", index=False)


Total shots: 22356
Shots available for modeling: 18018
Shots dropped (first 3 per game): 4338


      gameid  period        clock  h_pts  a_pts team  playerid    player          type   subtype result  x  y  dist                  desc  season  outcome
31  20900030       2  PT00M52.40S    0.0    0.0  GSW    201939  S. Curry          Foul  Shooting    NaN  0  0     0  Curry S.FOUL (P3.PN)    2010        0
32  20900030       2  PT00M52.40S    0.0    0.0  GSW    201939  S. Curry  Substitution       NaN    NaN  0  0     0  SUB: Ellis FOR Curry    2010        0


In [11]:
import statsmodels.formula.api as smf

shots['lag1'] = shots.groupby(['player_id','game_id'])['outcome'].shift(1)
shots['lag2'] = shots.groupby(['player_id','game_id'])['outcome'].shift(2)
shots['lag3'] = shots.groupby(['player_id','game_id'])['outcome'].shift(3)
shots = shots.dropna(subset=['lag1','lag2','lag3'])

model = smf.logit(
    'outcome ~ lag1 + lag2 + lag3 + shot_distance + defender_distance + C(shot_type)',
    data=shots
).fit(cov_type='cluster', cov_kwds={'groups': shots['player_id']})

print(model.summary())

NameError: name 'shots' is not defined